# Train Path_GRU on Real-IM (Kaggle GPU)

Before running: **Settings → Accelerator → GPU T4 x2**, **Settings → Internet → On**,
and **Add Input** → your Real-IM dataset (and, when resuming, the previous run's output).

Run it with **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed.
Everything under `/kaggle/working` (checkpoints, logs) becomes this version's output.
See `SETUP.md` §10 in the repo for the full walkthrough.

In [ ]:
# ===== Parameters: the only cell you normally edit =====
REPO_URL = "https://github.com/<your-github-user>/iros2024_poseForecasting.git"
BRANCH = "main"
USE_GITHUB_TOKEN = False      # True for a private repo: add a Kaggle Secret named GITHUB_TOKEN

CFG = "real_Path_GRU"         # a cfg/<name>.yml in the repo
EXP = "kaggle_v1"             # run name -> results/<CFG>_<EXP>/   (never "exp": that's the pretrained run)
START_EPOCH = 0               # 0 = from scratch; N = resume from pathnet_N.pt / posenet_N.pt in an attached input
RESUME_FROM_EXP = EXP         # which run the resume checkpoints come from ("exp" = the published weights)
N_EPOCHS = 600                # epochs to run in THIS session (keep START_EPOCH a multiple of 600, see LR note)
BATCH_SIZE = 32
SMOKE_TEST = True             # True: run 2 epochs only, to measure speed. Set False for the real run.
TIME_LIMIT_H = 11             # stop training cleanly before Kaggle's 12 h session limit

BASE_LR, LR_STEP = 1e-4, 600  # must match lr / lr_step_size in the cfg
KEEP_ONLY_LAST_CKPT = False   # True: delete all but the newest checkpoint pair at the end (saves output space)

In [ ]:
# ===== 1. Check the GPU =====
import sys, subprocess, torch
print(sys.version)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2 (needs a phone-verified account)."
cap = torch.cuda.get_device_capability(0)
arch = f"sm_{cap[0]}{cap[1]}"
print(torch.cuda.get_device_name(0), arch, "| torch built for:", torch.cuda.get_arch_list())
assert arch in torch.cuda.get_arch_list(), (
    f"This torch build has no kernels for {arch} (the P100 is sm_60, dropped by recent builds). "
    "Switch the accelerator to GPU T4 x2.")

In [ ]:
# ===== 2. Get the code and the one missing package =====
import os, glob, shutil
REPO = "/kaggle/working/iros2024_poseForecasting"
url = REPO_URL
if USE_GITHUB_TOKEN:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # keep the token out of the output
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit)

# open3d is only used for visualisation, but the loader imports it at module level.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open3d"], check=True)
import open3d; print("open3d", open3d.__version__)

In [ ]:
# ===== 3. Link the dataset where the cfg expects it (data/Real-IM-Dataset-processed/) =====
hits = glob.glob("/kaggle/input/**/kl_f2_seq1_raw.npz", recursive=True)
assert hits, "Real-IM dataset not found: Add Input -> Your Work -> Datasets -> your Real-IM dataset"
data_src = os.path.dirname(hits[0])
print("data:", data_src, sorted(os.listdir(data_src)))
data_dst = f"{REPO}/data/Real-IM-Dataset-processed"
os.makedirs(os.path.dirname(data_dst), exist_ok=True)
if os.path.islink(data_dst):
    os.unlink(data_dst)
os.symlink(data_src, data_dst)

# ===== 4. Resume: copy pathnet_/posenet_<START_EPOCH>.pt from an attached input =====
run_dir = f"{REPO}/results/{CFG}_{EXP}"
models_dir = f"{run_dir}/models"
os.makedirs(models_dir, exist_ok=True)
open(f"{run_dir}/code_commit.txt", "a").write(f"{commit}  start_epoch={START_EPOCH}\n")
resumed_files = []
if START_EPOCH > 0:
    for net in ("pathnet", "posenet"):
        name = f"{net}_{START_EPOCH:03d}.pt"
        cands = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if len(cands) > 1:
            cands = [c for c in cands if f"{CFG}_{RESUME_FROM_EXP}" in c] or cands
        assert len(cands) == 1, f"need exactly one {name} under /kaggle/input, found: {cands}"
        shutil.copy(cands[0], f"{models_dir}/{name}")
        resumed_files.append(f"{models_dir}/{name}")
        print("resume from", cands[0])

In [ ]:
# ===== 5. Train =====
import time
epochs = 2 if SMOKE_TEST else N_EPOCHS
# train.py restarts its StepLR schedule on every run, so pass the LR this epoch should have.
lr = BASE_LR * 0.1 ** (START_EPOCH // LR_STEP)
if START_EPOCH % LR_STEP:
    print(f"WARNING: START_EPOCH={START_EPOCH} is not a multiple of {LR_STEP}; LR drops will be shifted.")
cmd = (f"set -o pipefail; cd {REPO} && TQDM_MININTERVAL=30 timeout {TIME_LIMIT_H}h "
       f"{sys.executable} -u train.py --cfg {CFG} --exp {EXP} --start_epoch {START_EPOCH} --epoch {epochs} "
       f"--batch_size {BATCH_SIZE} --lr {lr} --train_pathnet --train_posenet "
       f"2>&1 | tee -a {run_dir}/train_log.txt")
print(cmd)
t0 = time.time()
rc = subprocess.run(["bash", "-c", cmd]).returncode
dt = time.time() - t0
print(f"\nexit code {rc}, {dt/3600:.2f} h, ~{dt/epochs:.1f} s/epoch (incl. data loading)")
if rc == 124:
    print("Stopped by the TIME_LIMIT_H guard. Resume from the last saved checkpoint (cell 7 lists them).")
elif rc != 0:
    raise RuntimeError("train.py failed, see the log above")
if SMOKE_TEST:
    per_epoch = dt / epochs
    print(f"Estimate: {per_epoch*3600/3600:.1f} h for 3600 epochs; "
          f"~{int(TIME_LIMIT_H*3600/per_epoch)} epochs fit in {TIME_LIMIT_H} h (first epoch includes startup, so this is pessimistic).")

In [ ]:
# ===== 6. Evaluate the newest checkpoint on the test sequence (cfg 'test' list) =====
import re
saved = sorted(int(re.search(r"pathnet_(\d+)\.pt", p).group(1)) for p in glob.glob(f"{models_dir}/pathnet_*.pt"))
print("checkpoints in this run:", saved)
new = [e for e in saved if e > START_EPOCH]
if new:
    last = new[-1]
    subprocess.run(["bash", "-c", f"set -o pipefail; cd {REPO} && {sys.executable} -u eval.py --cfg {CFG} --exp {EXP} "
                    f"--epoch {last} 2>&1 | grep -v '%|' | tee -a {run_dir}/eval_log_{last}.txt"], check=True)
else:
    print("no new checkpoint this session (smoke test, or fewer than save_freq epochs): skipping eval")

In [ ]:
# ===== 7. Clean the output: only what's needed for the next session is kept =====
os.unlink(data_dst)                                   # don't save a link to /kaggle/input
for f in resumed_files:                               # already in the previous version's output
    if new: os.remove(f)
if KEEP_ONLY_LAST_CKPT and new:
    for e in saved:
        if e != new[-1]:
            for net in ("pathnet", "posenet"):
                p = f"{models_dir}/{net}_{e:03d}.pt"
                if os.path.exists(p): os.remove(p)
subprocess.run(["bash", "-c", f"du -sh /kaggle/working; ls -la {models_dir}"])